# NB00 — Setup, access check and SQL dry-run

**Purpose.** Confirm the Drive folder layout, BigQuery access, and validate **every** extraction query with a BigQuery *dry run* (no data is read, no cost). Nothing downstream should be run until every query shows `ok = True`.



| | |
|---|---|
| **Inputs** | BigQuery access to `physionet-data` |
| **Outputs** | `results/NB00/` (dry-run table, schema probes) |
| **Free-Colab runtime** | ~5 min |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

### Before you start
1. Upload the whole local `Plan A/v4` folder to Google Drive so that this path exists:
   `MyDrive/Plan A/v4/code/transport_utils.py`
2. Check the `PROJECT` name in the BigQuery cell below (your old notebooks used `mimic-rs`).

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB00')

In [ ]:
# Folder check
for d in ['code', 'notebooks', 'data/raw', 'data/analysis', 'models', 'results']:
    p = os.path.join(BASE, d)
    os.makedirs(p, exist_ok=True)
    print(f"{'OK ' if os.path.isdir(p) else 'MISSING'}  {p}")
assert os.path.exists(os.path.join(BASE, 'code', 'sql_queries.py')), 'Upload code/sql_queries.py first'
import sklearn, lightgbm, xgboost, statsmodels
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__,
      '| xgboost', xgboost.__version__, '| pandas', pd.__version__)

In [ ]:
# ==== BigQuery connection ====
from google.colab import auth
from google.cloud import bigquery
auth.authenticate_user()
PROJECT = 'mimic-rs'          # <-- your Google Cloud project with PhysioNet access
client = bigquery.Client(project=PROJECT)
import sql_queries as sq
importlib.reload(sq)
print('Connected to BigQuery project:', PROJECT)

In [ ]:
# 1) Tiny access test on both databases
for q in ["SELECT COUNT(*) AS n FROM `physionet-data.mimiciv_3_1_icu.icustays`",
          "SELECT COUNT(*) AS n FROM `physionet-data.eicu_crd.hospital`"]:
    try:
        print(client.query(q).to_dataframe().iloc[0, 0], '<-', q)
    except Exception as ex:
        print('FAILED:', q, '\n', str(ex)[:400])

In [ ]:
# 2) Dry-run every extraction query (validates SQL + column names, reports GB scanned, costs nothing)
dr = pd.DataFrame(sq.dry_run_all(client))
tu.save_table(dr, 'dry_run')
print('\nTotal GB that the extraction will scan:', dr['gb_processed'].sum())
if not dr['ok'].all():
    print('\n>>> Some queries FAILED. Check the failing query (error text below) before continuing.')
    for _, r in dr[~dr['ok']].iterrows():
        print('\n---', r['name'], '---\n', r['error'])

In [ ]:
# 2b) If ONLY the mimic_* queries failed with a DATETIME/TIMESTAMP type error, run this cell once.
if not dr.loc[dr['name'].str.startswith('mimic'), 'ok'].all():
    sq.MIMIC_TIME = 'TIMESTAMP'
    dr2 = pd.DataFrame(sq.dry_run_all(client, [n for n in sq.QUERIES if n.startswith('mimic')]))
    print(dr2)
    if dr2['ok'].all():
        print("\nTIMESTAMP works -> open code/sql_queries.py and set MIMIC_TIME = 'TIMESTAMP'")

In [ ]:
# 3) Schema probes that the design depends on (cheap queries)
probes = {
 'eicu_year_distribution': """SELECT hospitaldischargeyear AS year, COUNT(*) AS n
                               FROM `physionet-data.eicu_crd.patient` GROUP BY 1 ORDER BY 1""",
 'eicu_apache_versions': """SELECT apacheversion, COUNT(*) AS n,
                             COUNTIF(SAFE_CAST(predictedhospitalmortality AS FLOAT64) >= 0) AS n_valid
                             FROM `physionet-data.eicu_crd.apachepatientresult` GROUP BY 1""",
 'eicu_nc_bp_names': """SELECT nursingchartcelltypevalname AS name, COUNT(*) AS n
                         FROM `physionet-data.eicu_crd.nursecharting`
                         WHERE nursingchartcelltypevallabel IN ('Non-Invasive BP','Invasive BP','Temperature')
                         GROUP BY 1 ORDER BY n DESC LIMIT 20""",
 'mimic_anchor_year_groups': """SELECT anchor_year_group, COUNT(*) AS n
                                 FROM `physionet-data.mimiciv_3_1_hosp.patients` GROUP BY 1 ORDER BY 1""",
}
probe_out = {}
for name, q in probes.items():
    try:
        df = client.query(q).to_dataframe()
        print(f'\n=== {name} ===\n', df.to_string(index=False))
        tu.save_table(df, f'probe_{name}', show=False)
        probe_out[name] = df.to_dict('records')
    except Exception as ex:
        print(f'\n=== {name} FAILED ===', str(ex)[:300])
        probe_out[name] = 'FAILED: ' + str(ex)[:300]

In [ ]:
tu.finish_notebook({
    'all_queries_ok': bool(dr['ok'].all()),
    'dry_run': dr.to_dict('records'),
    'probes': probe_out,
})